# Dashboard 3W - Analise de Features e Classes
**TCC:** Sistema para Deteccao e Classificacao de Eventos em Pocos Offshore

Execute **Kernel > Restart & Run All** para gerar todos os graficos.

In [ ]:
%matplotlib inline
import os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 12, 'axes.labelsize': 10})

PROJECT_ROOT = Path(os.environ.get('PROJECT_DIR', '/home/jovyan/workspace'))
REPORT_DIR   = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

SENSOR_COLUMNS = ['P-PDG','P-TPT','T-TPT','P-MON-CKP','T-JUS-CKP','P-JUS-CKGL','T-JUS-CKGL','QGL']
EVENT_LABELS   = {0:'Normal',1:'BSW Abrupt',2:'DHSV Closure',3:'Severe Slug',
                  4:'Flow Instab.',5:'Prod. Loss',6:'Restriction',7:'Scaling',
                  8:'Hydrate',9:'Undefined'}
COLORS = plt.cm.tab10.colors
print('Setup OK | PROJECT_ROOT:', PROJECT_ROOT)

In [ ]:
def load_parquets(path, pattern='*.parquet'):
    files = list(path.glob(pattern))
    if not files:
        print(f'[AVISO] Sem dados em: {path}')
        return pd.DataFrame()
    df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
    print(f'{path.name}: {len(df):,} linhas | {df.shape[1]} colunas')
    return df

df_silver = load_parquets(PROJECT_ROOT / 'data' / 'silver')
df_train  = load_parquets(PROJECT_ROOT / 'data' / 'gold', 'train.parquet')
df_val    = load_parquets(PROJECT_ROOT / 'data' / 'gold', 'val.parquet')
df_test   = load_parquets(PROJECT_ROOT / 'data' / 'gold', 'test.parquet')

if not df_silver.empty and 'class' in df_silver.columns:
    df_silver['class'] = df_silver['class'].astype(int)

SC = [c for c in SENSOR_COLUMNS if not df_silver.empty and c in df_silver.columns]
print(f'Sensores: {SC}')

---
## 1. Distribuicao de Classes

In [ ]:
if df_silver.empty or 'class' not in df_silver.columns:
    print('Sem dados Silver.')
else:
    counts = df_silver['class'].value_counts().sort_index()
    total  = counts.sum()
    labels = [f'C{k}\n{EVENT_LABELS.get(k,"?")}' for k in counts.index]
    colors = [COLORS[k % 10] for k in counts.index]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    bars = axes[0].barh(labels, counts.values, color=colors, edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, counts.values):
        axes[0].text(bar.get_width() * 1.01, bar.get_y() + bar.get_height()/2,
                     f'{val:,}  ({val/total*100:.1f}%)', va='center', fontsize=8)
    axes[0].set_xlabel('Registros')
    axes[0].set_title('Contagem por Classe (Silver)')
    axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    axes[0].invert_yaxis()

    axes[1].pie(counts.values, labels=[f'C{k}' for k in counts.index], colors=colors,
                autopct='%1.1f%%', startangle=90,
                wedgeprops=dict(edgecolor='white', linewidth=0.8))
    axes[1].set_title(f'Proporcao por Classe\nImbalance ratio: {counts.max()/counts.min():.1f}x')

    plt.suptitle('Distribuicao de Classes de Evento', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(REPORT_DIR / 'dash_01_classes.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Total: {total:,} | Classes: {len(counts)}')

---
## 2. Distribuicao dos Sensores por Classe (Boxplots)

In [ ]:
if not df_silver.empty and SC:
    classes  = sorted(df_silver['class'].unique())
    n_sensors = len(SC)
    ncols = 2
    nrows = (n_sensors + 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 3.2))
    axes = axes.flatten()

    for idx, sensor in enumerate(SC):
        ax    = axes[idx]
        data  = []
        xlbls = []
        clrs  = []
        for cls in classes:
            vals = df_silver.loc[df_silver['class'] == cls, sensor].dropna()
            if len(vals) < 5:
                continue
            # .to_numpy() evita o ValueError de dimensoes com pandas Series
            data.append(vals.sample(min(800, len(vals)), random_state=42).to_numpy())
            xlbls.append(f'C{cls}')
            clrs.append(COLORS[cls % 10])

        if not data:
            ax.set_visible(False)
            continue

        bp = ax.boxplot(data, labels=xlbls, patch_artist=True,
                        medianprops=dict(color='black', linewidth=1.5),
                        flierprops=dict(marker='.', markersize=1.5, alpha=0.3),
                        whiskerprops=dict(linewidth=0.8),
                        capprops=dict(linewidth=0.8))
        for patch, color in zip(bp['boxes'], clrs):
            patch.set_facecolor(color)
            patch.set_alpha(0.65)
        ax.set_title(sensor, fontweight='bold')
        ax.set_xlabel('Classe')
        ax.grid(axis='y', alpha=0.3)

    for j in range(idx + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Distribuicao dos Sensores por Classe (Silver)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(REPORT_DIR / 'dash_02_boxplots.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 3. Matriz de Correlacao dos Sensores

In [ ]:
if not df_silver.empty and SC:
    sample = df_silver[SC].sample(min(10000, len(df_silver)), random_state=42)
    corr   = sample.corr()

    fig, ax = plt.subplots(figsize=(9, 7))
    diag_mask = np.eye(len(corr), dtype=bool)
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
                vmin=-1, vmax=1, center=0, square=True,
                linewidths=0.5, ax=ax, mask=diag_mask,
                annot_kws={'size': 9})
    ax.set_title('Matriz de Correlacao dos Sensores (Silver)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(REPORT_DIR / 'dash_03_correlacao.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('Pares com |correlacao| > 0.7:')
    shown = False
    for i in range(len(corr)):
        for j in range(i+1, len(corr)):
            v = corr.iloc[i, j]
            if abs(v) > 0.7:
                print(f'  {corr.columns[i]} <-> {corr.index[j]}: {v:.3f}')
                shown = True
    if not shown:
        print('  Nenhum par com |correlacao| > 0.7')

---
## 4. Media e Desvio Padrao dos Sensores por Classe

In [ ]:
if not df_silver.empty and SC:
    for metric_fn, metric_name in [('mean','Media'), ('std','Desvio Padrao')]:
        class_stats = df_silver.groupby('class')[SC].agg(metric_fn)

        fig, ax = plt.subplots(figsize=(13, 4))
        x         = np.arange(len(SC))
        n_classes = len(class_stats)
        bar_w     = 0.8 / max(n_classes, 1)

        for i, (cls, row) in enumerate(class_stats.iterrows()):
            offset = (i - n_classes / 2 + 0.5) * bar_w
            ax.bar(x + offset, row[SC].values, bar_w * 0.9,
                   color=COLORS[cls % 10], alpha=0.8,
                   label=f'C{cls}: {EVENT_LABELS.get(cls, "?")}')

        ax.set_xticks(x)
        ax.set_xticklabels(SC, rotation=20, ha='right')
        ax.set_title(f'{metric_name} dos Sensores por Classe (Silver)', fontweight='bold')
        ax.set_ylabel(metric_name)
        ax.legend(loc='upper right', fontsize=7, ncol=2,
                  bbox_to_anchor=(1.15, 1), borderaxespad=0)
        ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.savefig(REPORT_DIR / f'dash_04_{metric_fn}.png', dpi=150, bbox_inches='tight')
        plt.show()

---
## 5. Poder Discriminativo dos Sensores

In [ ]:
if not df_silver.empty and SC:
    class_means = df_silver.groupby('class')[SC].mean()
    discrimin   = class_means.std().sort_values(ascending=True)
    median_val  = discrimin.median()
    colors_d    = ['#d62728' if v >= median_val else '#1f77b4' for v in discrimin.values]

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.barh(discrimin.index, discrimin.values, color=colors_d, edgecolor='white')
    ax.axvline(median_val, color='gray', linestyle='--', linewidth=1.2,
               label=f'Mediana ({median_val:.3f})')
    for bar, val in zip(bars, discrimin.values):
        ax.text(val + discrimin.max() * 0.01, bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', fontsize=8)
    ax.set_xlabel('Desvio padrao das medias por classe')
    ax.set_title('Poder Discriminativo dos Sensores\n(maior = mais util para classificacao)',
                 fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(REPORT_DIR / 'dash_05_discriminativo.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Melhor sensor: {discrimin.idxmax()} ({discrimin.max():.4f})')
    print(f'Pior  sensor : {discrimin.idxmin()} ({discrimin.min():.4f})')

---
## 6. Nulos por Sensor (Silver)

In [ ]:
if not df_silver.empty and SC:
    null_pct = (df_silver[SC].isnull().mean() * 100).round(2)
    clrs = ['#d62728' if v > 5 else '#2ca02c' for v in null_pct.values]

    fig, ax = plt.subplots(figsize=(11, 3.5))
    bars = ax.bar(null_pct.index, null_pct.values, color=clrs, edgecolor='white')
    ax.axhline(5, color='red', linestyle='--', linewidth=1, label='Limite 5%')
    for bar, val in zip(bars, null_pct.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=8)
    ax.set_ylabel('% Nulos')
    ax.set_title('Percentual de Valores Nulos por Sensor (Silver)', fontweight='bold')
    ax.set_ylim(0, max(null_pct.max() * 1.4, 12))
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(REPORT_DIR / 'dash_06_nulos.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 7. Splits da Camada Gold

In [ ]:
if not df_train.empty and 'class' in df_train.columns:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    splits = [('Treino (70%)', df_train), ('Validacao (15%)', df_val), ('Teste (15%)', df_test)]

    for ax, (name, df_sp) in zip(axes, splits):
        if df_sp.empty:
            ax.set_visible(False)
            continue
        counts = df_sp['class'].astype(int).value_counts().sort_index()
        clrs   = [COLORS[k % 10] for k in counts.index]
        bars   = ax.bar([f'C{k}' for k in counts.index], counts.values,
                        color=clrs, edgecolor='white')
        for bar, val in zip(bars, counts.values):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + counts.max() * 0.01,
                    f'{val:,}', ha='center', va='bottom', fontsize=7)
        ax.set_title(f'{name}\n{len(df_sp):,} registros', fontweight='bold')
        ax.set_ylabel('Registros')
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
        ax.grid(axis='y', alpha=0.3)

    plt.suptitle('Distribuicao por Classe em Cada Split (Gold)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(REPORT_DIR / 'dash_07_gold_splits.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 8. Distribuicao Z-score das Features (Gold)

In [ ]:
if not df_train.empty:
    base_norm = [c for c in df_train.columns
                 if c.endswith('_norm') and
                 not any(x in c for x in ['roll_', 'lag_', 'delta'])][:8]

    if not base_norm:
        print('[AVISO] Nenhuma feature base normalizada encontrada na camada Gold.')
    else:
        fig, axes = plt.subplots(2, 4, figsize=(15, 6))
        axes = axes.flatten()
        for i, feat in enumerate(base_norm):
            ax = axes[i]
            for df_sp, name, color in [
                (df_train, 'Treino', '#1f77b4'),
                (df_val,   'Val',    '#ff7f0e'),
                (df_test,  'Teste',  '#2ca02c'),
            ]:
                if df_sp.empty or feat not in df_sp.columns:
                    continue
                ax.hist(df_sp[feat].dropna().to_numpy(), bins=50, alpha=0.5,
                        color=color, label=name, density=True, edgecolor='none')
            ax.set_title(feat.replace('_norm', '').replace('_', '-').upper(), fontsize=9)
            ax.set_xlabel('Z-score', fontsize=8)
            ax.tick_params(labelsize=7)
            ax.grid(alpha=0.2)
            if i == 0:
                ax.legend(fontsize=7)
        for j in range(len(base_norm), len(axes)):
            axes[j].set_visible(False)
        plt.suptitle('Distribuicao Z-score das Features Base (Gold)', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.savefig(REPORT_DIR / 'dash_08_features_norm.png', dpi=150, bbox_inches='tight')
        plt.show()

---
## 9. Serie Temporal - Poco com Mais Classes

In [ ]:
if not df_silver.empty and 'timestamp' in df_silver.columns and 'ID_Poco' in df_silver.columns:
    df_silver['timestamp'] = pd.to_datetime(df_silver['timestamp'])
    poco  = df_silver.groupby('ID_Poco')['class'].nunique().idxmax()
    df_p  = df_silver[df_silver['ID_Poco'] == poco].sort_values('timestamp').copy()

    if len(df_p) > 30000:
        step = len(df_p) // 30000 + 1
        df_p = df_p.iloc[::step]

    sensors_plot = [c for c in ['P-PDG', 'P-TPT', 'T-TPT', 'QGL'] if c in df_p.columns]
    classes_p    = sorted(df_p['class'].unique())
    n = len(sensors_plot)
    if n == 0:
        print('Nenhum sensor disponivel para plot.')
    else:
        fig, axes = plt.subplots(n, 1, figsize=(15, n * 2.5), sharex=True)
        if n == 1:
            axes = [axes]
        for ax, sensor in zip(axes, sensors_plot):
            for cls in classes_p:
                mask = df_p['class'] == cls
                ax.scatter(df_p.loc[mask, 'timestamp'], df_p.loc[mask, sensor],
                           s=0.8, alpha=0.5, color=COLORS[cls % 10],
                           label=f'C{cls}: {EVENT_LABELS.get(cls, "?")}')
            ax.set_ylabel(sensor, fontsize=9)
            ax.legend(loc='upper right', fontsize=6, markerscale=5,
                      ncol=min(4, len(classes_p)))
            ax.grid(alpha=0.2)
        axes[-1].set_xlabel('Timestamp')
        fig.suptitle(f'Serie Temporal - Poco {poco}', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.savefig(REPORT_DIR / 'dash_09_timeseries.png', dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Poco: {poco} | Classes: {[int(c) for c in classes_p]}')

---
## 10. Resumo Executivo

In [ ]:
print('=' * 60)
print(' RESUMO EXECUTIVO - Dashboard 3W')
print('=' * 60)

if not df_silver.empty and 'class' in df_silver.columns:
    counts   = df_silver['class'].value_counts()
    null_max = df_silver[SC].isnull().mean().max() * 100 if SC else 0
    best     = df_silver.groupby('class')[SC].mean().std().idxmax() if SC else 'N/A'
    print(f'\n[Silver]')
    print(f'  Registros      : {len(df_silver):,}')
    print(f'  Classes        : {sorted(df_silver["class"].unique().tolist())}')
    print(f'  Imbalance ratio: {counts.max()/counts.min():.1f}x')
    print(f'  Classe maior   : C{int(counts.idxmax())} {EVENT_LABELS.get(int(counts.idxmax()),"?")} ({counts.max():,})')
    print(f'  Nulos max      : {null_max:.1f}%')
    print(f'  Melhor sensor  : {best}')

if not df_train.empty:
    norm_cols = [c for c in df_train.columns if c.endswith('_norm')]
    total     = len(df_train) + len(df_val) + len(df_test)
    print(f'\n[Gold - ML Ready]')
    print(f'  Total          : {total:,}')
    print(f'  Features       : {len(norm_cols)}')
    print(f'  Treino/Val/Test: {len(df_train):,} / {len(df_val):,} / {len(df_test):,}')

reports = list(REPORT_DIR.glob('dash_*.png'))
print(f'\n[Graficos em reports/]: {len(reports)} arquivos')
print('\n[Proximas etapas]')
print('  1. Treinar Random Forest (baseline)')
print('  2. Treinar XGBoost e LightGBM')
print('  3. Comparar F1-weighted')
print('  4. Avaliar com matriz de confusao')
print('=' * 60)